In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import numpy as np
import pandas as pd
from scipy.signal import resample_poly
from IPython.display import display

In [ ]:
PROJECT_DIR = Path("/content/drive/MyDrive/SleepStageClassificationProject")
OLD_DIR = PROJECT_DIR / "dataset" / "preprocess_microvolt"
NEW_DIR = PROJECT_DIR / "dataset" / "preprocess_microvolt_multi5"

SPLIT = "train"
N_SUBJECTS = 3

ALL_CHANNELS = ["F4-M1", "C4-M1", "O2-M1", "C3-M2", "E1-M2", "E2-M2", "Chin"]
CH7 = ["F4-M1", "C4-M1", "O2-M1", "C3-M2", "E1-M2", "E2-M2", "Chin"]
CH4 = ["C4-M1", "C3-M2", "E1-M2", "Chin"]

STAGES = ["W", "N1", "N2", "N3", "R"]

In [ ]:
for name, folder in [("old", OLD_DIR), ("new", NEW_DIR)]:
    path = sorted((folder / SPLIT).glob("*.npz"))[0]
    print(name, path.name)
    with np.load(path, allow_pickle=True) as data:
        for key in data.files:
            if key in ["subject_id", "stage"]:
                continue
            print("  ", key, data[key].shape)
    print()

# ข้อมูลดิบใน npz
# X (N_seq, N_epoch_per_seq, N_sample, N_ch)   เช่น (138, 5, 7680, 7)
# y (N_seq, N_epoch_per_seq)                   เช่น (138, 5)

In [ ]:
def load_data(folder, split, n_subjects):
    X_list = []
    rows = []
    count = 0

    for path in sorted((folder / split).glob("*.npz")):
        if count >= n_subjects:
            break

        sub_X = []
        sub_rows = []
        data = np.load(path, allow_pickle=True)

        for g in range(len(data["stage"])):
            stage = str(data["stage"][g])
            if stage not in STAGES or int(data["n_valid"][g]) != 5:
                continue

            y_group = np.asarray(data["y"][g]).astype(str).ravel()
            epochs = np.asarray(data["epoch_indices"][g]).astype(int).ravel()
            x = np.asarray(data["X"][g], dtype=np.float32)

            if not np.all(y_group == stage) or not np.all(np.diff(epochs) == 1):
                continue
            if x.shape != (5, 7680, 7) or not np.isfinite(x).all():
                continue

            x = x.reshape(-1, 7)
            x = resample_poly(x, 100, 256, axis=0)

            sub_X.append(x.astype(np.float32))
            sub_rows.append({
                "subject": path.stem,
                "stage": stage,
                "label": STAGES.index(stage),
                "epoch_start": int(epochs[0]),
                "epoch_end": int(epochs[-1]),
            })

        if len(set(r["stage"] for r in sub_rows)) == 5:
            X_list.extend(sub_X)
            rows.extend(sub_rows)
            count += 1

    meta = pd.DataFrame(rows)
    X = np.stack(X_list)
    y = meta["label"].to_numpy().astype(np.int32)
    return X, y, meta


X_old, y_old, meta_old = load_data(OLD_DIR, SPLIT, N_SUBJECTS)
X_new, y_new, meta_new = load_data(NEW_DIR, SPLIT, N_SUBJECTS)

print("old:", X_old.shape, y_old.shape)
print("new:", X_new.shape, y_new.shape)

# หลัง load_data (ป้อนเข้าโมเดล)
# X (N_seq, N_epoch_per_seq * N_sample, N_ch)  เช่น (138, 15000, 7)
# y (N_seq,)                                   เช่น (138,)

In [ ]:
def show(title, X_all, y, meta, channels):
    idx = [ALL_CHANNELS.index(c) for c in channels]
    X = X_all[:, :, idx]

    print("=" * 70)
    print(title, channels)
    print("=" * 70)

    print("X shape:", X.shape)
    print("X dtype:", X.dtype)
    print("X[0] shape:", X[0].shape)
    print("X[0] first 5 rows:")
    display(pd.DataFrame(X[0, :5, :], columns=channels))

    print("X[0] as array:")
    print(X[0, :5, :])

    print()
    print("y shape:", y.shape)
    print("y dtype:", y.dtype)
    print("y first 20:", y[:20])
    print("y as stage names:", [STAGES[i] for i in y[:20]])

    print()
    print("meta first 10 rows:")
    display(meta.head(10))

    print("sequences per subject and stage:")
    display(pd.crosstab(meta["subject"], meta["stage"])[STAGES])

    batch = np.random.permutation(len(y))[:8]
    print("batch X shape:", X[batch].shape)
    print("batch y:", y[batch])

In [ ]:
show("OLD data", X_old, y_old, meta_old, CH4)

In [ ]:
show("OLD data", X_old, y_old, meta_old, CH7)

In [ ]:
show("NEW data", X_new, y_new, meta_new, CH4)

In [ ]:
show("NEW data", X_new, y_new, meta_new, CH7)